# VQC Based on IQC:AIL

## Imports

In [1]:
import qiskit
from qiskit_machine_learning.algorithms import VQC
from qiskit.circuit import QuantumCircuit,Parameter
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram, visualize_transition, plot_bloch_vector
from qiskit.circuit.library import UnitaryGate,Initialize
from qiskit.quantum_info import Statevector,partial_trace, DensityMatrix

import pennylane as qml
from pennylane import numpy as pnp
import qutip
from toqito import state_props

import numpy as np
from scipy.linalg import expm as expMatrix
import scipy.linalg
from sympy.physics.quantum.dagger import Dagger
import math

from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold,train_test_split, KFold
from sklearn.datasets import make_moons
from sklearn.multiclass import OneVsRestClassifier
from sklearn.utils.multiclass import unique_labels
from sklearn.utils.validation import check_array, check_is_fitted
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn import preprocessing
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, make_scorer, roc_auc_score, classification_report

from ucimlrepo import fetch_ucirepo

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pdflatex

import pandas as pd


## Parâmetros

In [2]:
RANDOM_SEED = 1
N_SAMPLES = 1
N_PRINTINGS = 1
N_SHOTS=2048

## Tratamento do Dataset

In [3]:
def normalize_iqc_ail(data, normalize_col=False, normalize_lin=False):
    if normalize_col:
        scaler = MinMaxScaler() #Normaliza a coluna entre [0,1]
        scaler.fit(data)
        data = scaler.transform(data)
        '''
        Perceba que normalizando apenas a coluna, podemos ter amplitudes dos estados em que a norma do estado não fosse igual a 1. Para resolvermos isso, devemos
        normalizar as linhas entre si

        '''
        data = preprocessing.normalize(data,axis=1,norm='l2')
    if normalize_lin:
        data = preprocessing.normalize(data,axis=1,norm='l2') #Normaliza a linha entre [-1,1]
    return data
    
def normalize_iqc(data, normalize_col=False, normalize_lin=False):
    if normalize_col:
        scaler = MinMaxScaler() #Normaliza a coluna entre [0,1]
        scaler.fit(data)
        data = scaler.transform(data)
        data = data - 1
    if normalize_lin:
        data = preprocessing.normalize(data,axis=1,norm='l2') #Normalize the line between [-1,1]
    return data

## Circuito Teste

In [ ]:
N_features=4
N_QUBITS=math.ceil(np.log2(N_features)+1) #Nqubits do circuito
QUBITS=[i for i in range(N_QUBITS)]

qc = QuantumCircuit(N_QUBITS)
qc.h(QUBITS)

tx = np.random.rand(N_features)
tw = np.random.rand(N_features)
sigmaE = np.diag(tx)*tw.T

matriz_pauli_x=np.array([[0,1],[1,0]]) # Matriz de Pauli x
matriz_pauli_y=np.array([[0,-1j],[1j,0]]) # Matriz de Pauli y
matriz_pauli_z=np.array([[1,0],[0,-1]]) # Matriz de Pauli z

sigmaQ=matriz_pauli_x+matriz_pauli_y+matriz_pauli_z



#Operador Unitário
U=np.matrix(expMatrix(1j*np.kron(sigmaQ,sigmaE)))

# qubitstarget = [i for i in range(Ntarget)] - > Desnecessário agora, mas interessante para fazer a generalização
qc.unitary(U,QUBITS)

display(qc.draw('mpl'))

tqc=transpile(qc, optimization_level=3, basis_gates=["u3", "cx"], seed_transpiler=1)
display(tqc.draw('mpl'))

gate_val = 0
u3_dir = {}
for i, instruction in enumerate(tqc.data):
    if instruction.operation.name == 'u3':
        u3_dir['u3_'+str(gate_val)] = {'qubit':instruction.qubits[0], 'params': instruction.operation.params}
        gate_val +=1
        
print(u3_dir)
print()

gate_val = 0
u3_params = []
for i in range(len(u3_dir)):
    u3_params.append(u3_dir[f'u3_{i}']['params'])

array=np.array(u3_params)/np.pi
plt.plot(array)

plt.rcParams.update({'font.size': 10})
plt.rcParams.update({'figure.autolayout': True})
labels=[r'$\theta$',r'$\phi$',r'$\lambda$']
colors = ['forestgreen','darkorange','dodgerblue','deeppink' ]
fig,ax=plt.subplots(1,3,figsize=(15,5))

ax[0].hist(array[:,0],label=labels[0],color=colors[0],edgecolor='black')
ax[0].set_xlabel(f'Factor of $\pi$')
ax[0].set_ylabel('Frequency')
ax[0].legend()
ax[0].grid(linestyle='dashed')

ax[1].hist(array[:,1],label=labels[1],color=colors[1],edgecolor='black')
ax[1].set_xlabel(f'Factor of $\pi$')
ax[1].set_ylabel('Frequency')
ax[1].legend()
ax[1].grid(linestyle='dashed')

ax[2].hist(array[:,2],label=labels[2],color=colors[2],edgecolor='black')
ax[2].set_xlabel(f'Factor of $\pi$')
ax[2].set_ylabel('Frequency')
ax[2].legend()
ax[2].grid(linestyle='dashed')
plt.savefig(f'Histograma_parametros_NF{N_features}_.svg')
plt.show()

## Função que gera o Quantum Circuit

In [4]:
def blochvector(rho_cog,matriz_pauli_x,matriz_pauli_y,matriz_pauli_z):
    x_bloch = np.trace(matriz_pauli_x@rho_cog.data)
    y_bloch = np.trace(matriz_pauli_y@rho_cog.data)
    z_bloch = np.trace(matriz_pauli_z@rho_cog.data)
    return [x_bloch,y_bloch,z_bloch]
    
#Executar o circuito
def run_qasm_counts(qc, shots=N_SHOTS):
    qc.measure_all()
    qasm_simulator = Aer.get_backend("qasm_simulator")
    job = qasm_simulator.run(qc, shots=shots)
    result = job.result()
    return result.get_counts()

def cirq_iqc_ail(data,contador,w,counter,qubits, N_qubits,N_atributos,printar_cirq=False):

    X_moons_new=list(data)
    if np.log2(N_atributos)%2!=0 and np.log2(N_atributos)!=1:
        for k in range(2**(N_qubits-1) - N_atributos):
            w=np.append(w,0)
            X_moons_new=np.append(X_moons_new,0)
        sigmaE=np.diag(w)
    else:
        sigmaE=np.diag(w)
    
    #Podíamos inicializar assim pra facilitar as contas
    '''x=np.random.rand(2**N_atributos)
    w=np.random.rand(2**N_atributos)'''

    # IQC:AIL

    qc = QuantumCircuit(N_qubits)
    qc.initialize(X_moons_new, range(1,N_qubits))# Inicializaçao do estado inicial. Poderia ser qualquer estado.
    qc.h(0)



    #Montando os sigmas

    matriz_pauli_x=np.array([[0,1],[1,0]]) # Matriz de Pauli x
    matriz_pauli_y=np.array([[0,-1j],[1j,0]]) # Matriz de Pauli y
    matriz_pauli_z=np.array([[1,0],[0,-1]]) # Matriz de Pauli z

    sigmaQ=matriz_pauli_x+matriz_pauli_y+matriz_pauli_z

    

    #Operador Unitário
    U=np.matrix(expMatrix(1j*np.kron(sigmaQ,sigmaE)))

    # qubitstarget = [i for i in range(Ntarget)] - > Desnecessário agora, mas interessante para fazer a generalização
    qc.unitary(U,qubits)
    if counter==0:
        qc.draw("mpl", filename=f'./mpl_complete_U_NF{N_features}.svg')
    if printar_cirq==True:
        display(qc.draw('mpl')) #display(qc.draw("mpl", filename='./mpl_original.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos

    #qc.decompose().draw(output="mpl", style="clifford")
    tqc=transpile(qc, optimization_level=3, basis_gates=["u3", "cx"], seed_transpiler=1)

    gate_val = 0
    u3_dir = {}
    for i, instruction in enumerate(tqc.data):
        if instruction.operation.name == 'u3':
            u3_dir['u3_'+str(gate_val)] = {'qubit':instruction.qubits[0], 'params': instruction.operation.params}
            gate_val +=1
            
    if printar_cirq and dict(tqc.count_ops())['u3']<=50:
        print(u3_dir)
        print()

    gate_val = 0
    u3_params = []
    for i in range(len(u3_dir)):
        u3_params.append(u3_dir[f'u3_{i}']['params'])

    if dict(tqc.count_ops())['u3']<=50 and contador%N_PRINTINGS==0:
        tqc.draw("mpl", filename=f'./mpl_transpiled{contador}_NF{N_features}.svg')

    if printar_cirq==True and dict(tqc.count_ops())['u3']<=50:
        print(dict(tqc.count_ops()))
        display(tqc.draw('mpl')) #displat(qc.draw('mpl', filename='./mpl_transpile.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos

    # Mostrando o vetor de estado 
    sv = Statevector(qc)
    '''if contador%N_PRINTINGS==0:
        sv.draw("city", filename=f'./state_vector_city{contador}_NF{N_features}.svg')
        sv.draw("bloch", filename=f'./state_vector_bloch{contador}_NF{N_features}.svg')
        sv.draw("qsphere", filename=f'./state_vector_qsphere{contador}_NF{N_features}.svg')
    if printar_cirq==True:
        display(sv.draw("latex"))
    '''
    rho_cog = partial_trace(sv, qubits[1:])
    if printar_cirq==True:
        print(rho_cog)
    


    return blochvector(rho_cog,matriz_pauli_x,matriz_pauli_y,matriz_pauli_z),u3_params

def cirq_iqc(data,contador,w,counter, qubits, N_qubits,N_atributos,printar_cirq=False):

    X_iris_new=list(data)
    if np.log2(N_atributos)%2!=0 and np.log2(N_atributos)!=1:
        for k in range(2**(N_qubits-1) - N_atributos):
            w=np.append(w,0)
            X_iris_new=np.append(X_iris_new,0)
        sigmaE=np.diag(X_iris_new)*w.T
    else:
        sigmaE=np.diag(X_iris_new)*w.T
    
    #Podíamos inicializar assim pra facilitar as contas
    '''x=np.random.rand(2**N_atributos)
    w=np.random.rand(2**N_atributos)'''

    # IQC

    qc = QuantumCircuit(N_qubits)

    qc.h(0)
    qc.h(range(1,N_qubits))



    #Montando os sigmas

    matriz_pauli_x=np.array([[0,1],[1,0]]) # Matriz de Pauli x
    matriz_pauli_y=np.array([[0,-1j],[1j,0]]) # Matriz de Pauli y
    matriz_pauli_z=np.array([[1,0],[0,-1]]) # Matriz de Pauli z

    sigmaQ=matriz_pauli_x+matriz_pauli_y+matriz_pauli_z

    

    #Operador Unitário
    U=np.matrix(expMatrix(1j*np.kron(sigmaQ,sigmaE)))

    # qubitstarget = [i for i in range(Ntarget)] - > Desnecessário agora, mas interessante para fazer a generalização
    qc.unitary(U,qubits)
    if counter==0:
        qc.draw("mpl", filename=f'./mpl_complete_U_iris_iqc.svg')
    if printar_cirq==True:
        display(qc.draw('mpl')) #display(qc.draw("mpl", filename='./mpl_original.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos

    #qc.decompose().draw(output="mpl", style="clifford")
    tqc=transpile(qc, optimization_level=3, basis_gates=["u3", "cx"], seed_transpiler=1)

    gate_val = 0
    u3_dir = {}
    for i, instruction in enumerate(tqc.data):
        if instruction.operation.name == 'u3':
            u3_dir['u3_'+str(gate_val)] = {'qubit':instruction.qubits[0], 'params': instruction.operation.params}
            gate_val +=1
            
    if printar_cirq and dict(tqc.count_ops())['u3']<=50:
        print(u3_dir)
        print()

    gate_val = 0
    u3_params = []
    for i in range(len(u3_dir)):
        u3_params.append(u3_dir[f'u3_{i}']['params'])

    if dict(tqc.count_ops())['u3']<=50 and contador%N_PRINTINGS==0:
        tqc.draw("mpl", filename=f'./mpl_transpiled{contador}_NF{N_features}.svg')

    if printar_cirq==True and dict(tqc.count_ops())['u3']<=50:
        print(dict(tqc.count_ops()))
        display(tqc.draw('mpl')) #displat(qc.draw('mpl', filename='./mpl_transpile.pdf')) #Trocar as chamadas se quiser salvar as imagens dos circuitos


    # Mostrando o vetor de estado 
    sv = Statevector(qc)
    '''if contador%N_PRINTINGS==0:
        sv.draw("city", filename=f'./state_vector_city{contador}_iris_iqc.svg')
        sv.draw("bloch", filename=f'./state_vector_bloch{contador}_iris_iqc.svg')
        sv.draw("qsphere", filename=f'./state_vector_qsphere{contador}_iris_iqc.svg')
    if printar_cirq==True:
        display(sv.draw("latex"))
        
    counts = run_qasm_counts(qc)
    if contador%N_PRINTINGS==0:
        plot_histogram(counts,filename=f'./histogram_plot_{contador}_iris_iqc.svg')'''

    rho_cog = partial_trace(sv, qubits[1:])
    if printar_cirq==True:
        print(rho_cog)

    
    return blochvector(rho_cog,matriz_pauli_x,matriz_pauli_y,matriz_pauli_z),u3_params

## Esfera de Bloch do Circuito

In [5]:
plt.rcParams.update({'font.size': 10})
plt.rcParams.update({'figure.autolayout': True})
labels=[r'$\theta$',r'$\phi$',r'$\lambda$']
colors = ['forestgreen','darkorange','dodgerblue','deeppink' ]

"""Definir X e w através de linspace e variar diretamente, sem construir a base de dados"""
def esfera_bloch_IQC_AIL(X,norma,weights,qubits, N_qubits,N_atributos,printar_esf=False):
    point_states=[]
    u3_params=[]
    for k in range(0,len(X)):
        bloch,params=cirq_iqc_ail(X[k],k,weights[k], qubits, N_qubits, N_atributos)
        point_states.append(bloch)
        u3_params.append(params)

    b = qutip.Bloch()
    b.point_default_color=['k']
    b.point_marker=['o']
    b.point_size=[15, 15, 15, 15]
    for k in range(len(point_states)):
        b.add_points(point_states[k])
    b.render()
    if printar_esf==True:
        b.show()

    bb = b.fig
    bb.savefig(f'Bloch_geral_NF{N_features}_IQC_AIL_{norma}.svg')
    return u3_params
    

def esfera_bloch_IQC(X,weights,norma,qubits, N_qubits,N_atributos,printar_esf=False):
    point_states=[]
    u3_params=[]
    for k in range(0,len(X)):
        bloch,params=cirq_iqc(X[k],k,weights[k], qubits, N_qubits, N_atributos)
        point_states.append(bloch)
        u3_params.append(params)


    b = qutip.Bloch()
    b.point_default_color=['k']
    b.point_marker=['o']
    b.point_size=[15, 15, 15, 15]
    for k in range(len(point_states)):
        b.add_points(point_states[k])
    b.render()
    if printar_esf==True:
        b.show()

    bb = b.fig
    bb.savefig(f'Bloch_geral_NF{N_features}_IQC_{norma}.svg')
    return u3_params

# PLOTAR OS HISTOGRAMAS
Para cada parâmetro $\theta, \phi, \lambda$ das P portas dos circuitos de um N_FEATURES, plotar um histograma através das N_SAMPLES

## Execuções

In [ ]:
# IQC:AIL
for N_features in [2,4,8]:
    X_df=np.random.rand(N_features,N_SAMPLES)
    w_df=np.random.rand(N_features,N_SAMPLES)
    N_QUBITS=math.ceil(np.log2(N_features)+1) #Nqubits do circuito
    QUBITS=[i for i in range(N_QUBITS)]
    X_df_iqc_ail_coluna=normalize_iqc_ail(X_df, normalize_col=True, normalize_lin=False)
    X_df_iqc_ail_linha=normalize_iqc_ail(X_df,normalize_col=False,normalize_lin=True)
    esfera_bloch_IQC_AIL(X_df_iqc_ail_coluna,w_df,'coluna',QUBITS, N_QUBITS,N_features)
    esfera_bloch_IQC_AIL(X_df_iqc_ail_linha,w_df,'linha',QUBITS, N_QUBITS,N_features)

In [ ]:
# IQC
for N_features in [2,4,8]:
    X_df=np.random.rand(N_SAMPLES,N_features)
    w_df=np.random.rand(N_SAMPLES,N_features)
    N_QUBITS=math.ceil(np.log2(N_features)+1) #Nqubits do circuito
    QUBITS=[i for i in range(N_QUBITS)]
    X_df_iqc_coluna=normalize_iqc(X_df, normalize_col=True, normalize_lin=False)
    X_df_iqc_linha=normalize_iqc(X_df,normalize_col=False,normalize_lin=True)
    esfera_bloch_IQC(X_df_iqc_coluna,w_df,'coluna',QUBITS, N_QUBITS,N_features)
    esfera_bloch_IQC(X_df_iqc_linha,w_df,'linha',QUBITS, N_QUBITS,N_features)